# Phase 4 (Colab, any NVIDIA GPU with 16 GB+): serving benchmarks

**Goal:** measure what each version of the model costs to run as a real service: how fast it answers one request, how many
requests per second it handles under load, how much GPU memory it needs, and the resulting **cost per 1M requests**.

**What is being compared** (all on the same GPU, same prompts, same settings):

| Name | Model file | Served with |
|---|---|---|
| `fp16` | merged 16-bit model (9 GB) | vLLM |
| `awq-default` | AWQ 4-bit, `linear_attn` kept 16-bit | vLLM |
| `awq-mlp-only` | AWQ 4-bit, MLP only | vLLM |
| `gguf-q4km` | GGUF 4-bit (2.7 GB) | llama.cpp |
| `gguf-q8` | GGUF 8-bit (4.5 GB) | llama.cpp |

**vLLM** is a serving engine built for GPUs and many simultaneous users. **llama.cpp** is a lighter program that runs GGUF
files on almost anything (we run it on the GPU here). Note that the GGUF rows differ from the others in *two* ways, the file
format and the serving program, so they compare complete "deployment stacks", not just the compression method.

**For each variant the notebook:** starts the server -> re-scores all 3,080 test examples *through the server* (a
final check that the quality numbers still hold in the real serving setup) -> sends load at 1, 4, 8, 16, 32, 64, 128 simultaneous
requests and records latency (p50/p95/p99), requests/s and tokens/s -> stops the server.

**Everything is written straight to your Google Drive** (`Banking_FInetuning_phase4/`) as it is produced, so a dropped session loses nothing.

**Before you run**
1. Runtime > Change runtime type > a GPU with 16 GB or more. **L4 is a good fit** for a 4B model (and the realistic choice for a deployment); A100 also works. T4 works for vLLM and llama.cpp, but vLLM's fast 4-bit kernels (Marlin) are not available on that older GPU generation (T4 is "Turing"), so the AWQ rows may be slow or fail there; the GGUF rows are fine on a T4.
2. Colab secret named `HF_TOKEN` (key icon in the left bar), your Hugging Face token with read access, with *Notebook access* turned on.
3. Have `banking-finetuning-code.zip` ready (the newest one). Put it in the root of your Drive, or you will be asked to upload it.

**Time:** about 1.5 to 2 hours in total (installs and downloads ~30 min, ~10 min per variant).

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')
import os, glob, shutil, subprocess, sys, json, time, threading, re, signal
import requests

RES = '/content/drive/MyDrive/Banking_FInetuning_phase4'
os.makedirs(RES, exist_ok=True)
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout

GPU_NAME, GPU_CC, GPU_MIB = [x.strip() for x in sh('nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader,nounits').strip().splitlines()[0].split(',')]
print('GPU:', GPU_NAME, '| compute capability', GPU_CC, '|', GPU_MIB, 'MiB')
assert float(GPU_CC) >= 7.5, 'vLLM needs compute capability 7.5 or higher (T4, L4, A10, A100, ...)'
assert int(GPU_MIB) >= 15000, 'Need a GPU with about 16 GB or more (the fp16 model alone is 9 GB)'
if float(GPU_CC) < 8.0:
    print('NOTE: this GPU is older than Ampere. The GGUF variants will work; the AWQ variants may be slow or fail in vLLM.')

# The project code: look on Drive first, otherwise ask for an upload.
cands = glob.glob('/content/drive/MyDrive/banking-finetuning-code.zip') + glob.glob('/content/drive/MyDrive/*/banking-finetuning-code.zip')
zpath = cands[0] if cands else list(files.upload())[0]
PROJ = '/content/Banking_FInetuning'
shutil.rmtree(PROJ, ignore_errors=True)
shutil.unpack_archive(zpath, PROJ)
os.chdir(PROJ)
assert os.path.exists('loadtest/bench_raw.py'), 'Old code zip: upload the newest banking-finetuning-code.zip'
print('code from', zpath, '| results go to', RES)
json.dump({'gpu': GPU_NAME, 'compute_cap': GPU_CC, 'memory_mib': int(GPU_MIB)}, open(f'{RES}/gpu.json', 'w'))

In [ ]:
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as e:
    raise RuntimeError('Add a Colab secret named HF_TOKEN (key icon, left bar) and enable Notebook access, then re-run this cell') from e
os.environ['HF_TOKEN'] = HF_TOKEN

from huggingface_hub import HfApi, snapshot_download, hf_hub_download
REPO = 'Abhishek-255/banking-triage-qwen3_5_4b-quantized'
repo_files = HfApi(token=HF_TOKEN).list_repo_files(REPO)
print(len(repo_files), 'files in', REPO)

# (name, engine, path inside the repo)
SPECS = [('fp16', 'vllm', 'merged-fp16'), ('awq-default', 'vllm', 'awq-default'), ('awq-mlp-only', 'vllm', 'awq-mlp-only'),
         ('gguf-q4km', 'llama', 'gguf/model-Q4_K_M.gguf'), ('gguf-q8', 'llama', 'gguf/model-Q8_0.gguf')]
def present(rel):
    if rel.endswith('.gguf'):
        return rel in repo_files
    return any(f.startswith(rel + '/') and f.endswith('.safetensors') for f in repo_files)
SPEC = {n: (k, r) for n, k, r in SPECS if present(r)}
print('available variants:', list(SPEC))
print('MISSING (will be skipped):', [n for n, _, r in SPECS if n not in SPEC])

### Install vLLM
vLLM brings its own PyTorch, so this takes a few minutes and Colab may print dependency warnings about its preinstalled
packages (harmless). If Colab shows a "restart session" button afterwards, press it and re-run from the first cell; files in
`/content` and on Drive survive a restart.

In [ ]:
!pip install -q vllm==0.30.0 httpx pandas scikit-learn pyyaml matplotlib
print(sh(sys.executable + ' -c "import vllm, transformers; print(vllm.__version__, transformers.__version__)"'))

## 1. Data
The held-out test set (3,080 examples, never used in training).

In [ ]:
if not os.path.exists('data/test.jsonl'):
    r = subprocess.run([sys.executable, '-m', 'src.data.prepare'], capture_output=True, text=True)
    print(r.stdout[-600:], r.stderr[-600:] if r.returncode else '')
print(sh('wc -l data/test.jsonl'))

## 2. Download the models from Hugging Face
About 25 GB, from your private repo. Files go to `/content/models` (the runtime disk, not Drive).

In [ ]:
MODELS = '/content/models'
os.makedirs(MODELS, exist_ok=True)
for name, (kind, rel) in SPEC.items():
    patt = [rel] if rel.endswith('.gguf') else [rel + '/*']
    print('downloading', name, '...')
    snapshot_download(REPO, allow_patterns=patt, local_dir=MODELS, token=HF_TOKEN)

# Qwen3.5 is a vision-language model, so vLLM expects the processor files next to the weights. The folders produced earlier
# may lack them, so copy the two small files from the original model if missing (weights are not touched).
def ensure_processor(d):
    for fname in ('preprocessor_config.json', 'video_preprocessor_config.json'):
        if not os.path.exists(f'{d}/{fname}'):
            hf_hub_download('Qwen/Qwen3.5-4B', fname, local_dir=d)
for name, (kind, rel) in SPEC.items():
    if kind == 'vllm':
        ensure_processor(f'{MODELS}/{rel}')
TOK = f'{MODELS}/merged-fp16' if 'fp16' in SPEC else 'Qwen/Qwen3.5-4B'   # the tokenizer used to render prompts
print('tokenizer for prompt rendering:', TOK)
print(sh(f'du -sh {MODELS}/* | head'))

## 3. Build llama.cpp (for the two GGUF variants)
Compiled for your GPU's architecture (detected automatically), about 10 minutes. The `libcuda.so` symlink is the same workaround that was needed on Kaggle (CMake wants a development link that GPU machines often lack). Skipped automatically if no GGUF variant is available.

In [ ]:
LLAMA, LLAMA_BIN = '/content/llama.cpp', '/content/llama.cpp/build/bin'
if any(k == 'llama' for k, _ in SPEC.values()):
    if not os.path.exists(LLAMA):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ggml-org/llama.cpp', LLAMA], check=True)
    cands = (glob.glob('/usr/local/cuda*/lib64/stubs/libcuda.so') + glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so*')
             + glob.glob('/usr/lib64-nvidia/libcuda.so*') + glob.glob('/usr/local/nvidia/lib64/libcuda.so*'))
    print('libcuda candidates:', cands)
    assert cands, 'No libcuda found: copy this message to Claude'
    STUB = '/content/cudastub'
    os.makedirs(STUB, exist_ok=True)
    if os.path.lexists(f'{STUB}/libcuda.so'):
        os.remove(f'{STUB}/libcuda.so')
    os.symlink(cands[0], f'{STUB}/libcuda.so')
    shutil.rmtree(f'{LLAMA}/build', ignore_errors=True)
    cfg = (f'cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={GPU_CC.replace(".", "")} -DCMAKE_LIBRARY_PATH={STUB} '
           f'-DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /content/cmake.log 2>&1')
    rc = subprocess.run(cfg, shell=True).returncode
    print('cmake configure exit code:', rc)
    if rc:
        print(sh("grep -n -i -B2 -A6 -E 'CMake Error|could not|not found|missing' /content/cmake.log | head -60"))
        raise RuntimeError('llama.cpp configure failed: copy the errors above to Claude')
    rc = subprocess.run(f'cmake --build {LLAMA}/build --target llama-server -j{os.cpu_count()} > /content/build.log 2>&1', shell=True).returncode
    print('build exit code:', rc)
    if rc:
        print(sh("grep -n -i -B3 -A3 -E 'error' /content/build.log | head -50"))
        raise RuntimeError('llama.cpp build failed: copy the errors above to Claude')
    assert os.path.exists(f'{LLAMA_BIN}/llama-server')
    print('built llama-server')
else:
    print('no GGUF variant available: skipping the llama.cpp build')

## 4. Helpers: start a server, measure, stop it
`VLLM_EXTRA` holds extra vLLM options. `--language-model-only` skips the (unused) vision part of Qwen3.5; if your vLLM version
rejects it, the error appears when the server starts and you can remove it here.

In [ ]:
PORT = 8000
VLLM_EXTRA = ['--language-model-only']
# GPU memory share vLLM may use: about 15 GiB in absolute terms (weights + working space), whatever the GPU size.
VLLM_UTIL = round(min(0.9, max(0.45, 15 * 1024 / int(GPU_MIB))), 2)
print('GPU:', GPU_NAME, '| vLLM memory share:', VLLM_UTIL)

def gpu_used_mib():
    return int(sh('nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits').strip().splitlines()[0])

def start_vllm(path, log_path):
    cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--model', path, '--served-model-name', 'banking',
           '--dtype', 'float16', '--max-model-len', '1024', '--gpu-memory-utilization', str(VLLM_UTIL), '--port', str(PORT)] + VLLM_EXTRA
    return subprocess.Popen(cmd, stdout=open(log_path, 'w'), stderr=subprocess.STDOUT, preexec_fn=os.setsid)

def start_llama(gguf, log_path):
    cmd = [f'{LLAMA_BIN}/llama-server', '-m', gguf, '--alias', 'banking', '--host', '127.0.0.1', '--port', str(PORT),
           '-ngl', '99', '-c', '65536', '-np', '64']
    return subprocess.Popen(cmd, stdout=open(log_path, 'w'), stderr=subprocess.STDOUT, preexec_fn=os.setsid)

def wait_ready(p, log_path, timeout=1500):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if p.poll() is not None:
            raise RuntimeError('server exited. Last log lines:' + chr(10) + sh(f'tail -40 {log_path}'))
        try:
            if requests.get(f'http://127.0.0.1:{PORT}/health', timeout=2).status_code == 200:
                return
        except Exception:
            pass
        time.sleep(3)
    raise RuntimeError('server not ready in time. Last log lines:' + chr(10) + sh(f'tail -40 {log_path}'))

def stop(p):
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        p.wait(timeout=40)
    except Exception:
        try:
            os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except Exception:
            pass
    sh('pkill -f vllm.entrypoints; pkill -f llama-server')
    for _ in range(40):                       # wait until the GPU memory is really free
        if gpu_used_mib() < 1500:
            return
        time.sleep(2)
    print('WARNING: GPU memory not fully released:', gpu_used_mib(), 'MiB')

class GpuMon(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True)
        self.halt, self.max = False, 0
    def run(self):
        while not self.halt:
            self.max = max(self.max, gpu_used_mib())
            time.sleep(0.5)

def parse_weights_gib(text):
    m = re.search(r'Model loading took ([0-9.]+)\s*GiB', text)                    # vLLM
    if m:
        return float(m.group(1))
    sizes = re.findall(r'model buffer size\s*=\s*([0-9.]+)\s*MiB', text)           # llama.cpp
    return round(sum(map(float, sizes)) / 1024, 2) if sizes else None

def stream(cmd):
    print('$', ' '.join(map(str, cmd)))
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if 'it/s' in line or 's/it' in line:
            continue
        print(line, end='')
    if p.wait():
        raise RuntimeError('command failed; see output above')

def run_variant(name):
    if name not in SPEC:
        print(f'{name}: not in the repo, skipping')
        return
    kind, rel = SPEC[name]
    d = f'{RES}/{name}'
    os.makedirs(d, exist_ok=True)
    log = f'{d}/server.log'
    path = f'{MODELS}/{rel}'
    idle = gpu_used_mib()
    p = start_vllm(path, log) if kind == 'vllm' else start_llama(path, log)
    mon = None
    try:
        t0 = time.time()
        wait_ready(p, log)
        load_s = round(time.time() - t0, 1)
        weights = parse_weights_gib(open(log, errors='ignore').read())
        print(f'{name}: server ready in {load_s}s | weights ~ {weights} GiB | GPU used {gpu_used_mib()} MiB')
        mon = GpuMon(); mon.start()
        # 1) quality, measured through the real serving stack (all 3,080 test examples)
        stream([sys.executable, '-m', 'src.eval.evaluate', '--model', 'banking', '--tokenizer', TOK, '--concurrency', 32,
                '--out', f'{d}/finetuned_test.json', '--preds-out', f'{d}/finetuned_test_preds.jsonl', '--tag', name])
        # 2) speed under load
        stream([sys.executable, 'loadtest/bench_raw.py', '--label', name, '--tokenizer', TOK, '--out-csv', f'{RES}/sweep.csv'])
        json.dump({'name': name, 'engine': kind, 'weights_gib': weights, 'load_seconds': load_s,
                   'gpu_used_mib_peak': mon.max, 'gpu_used_mib_idle_before': idle}, open(f'{d}/serve_info.json', 'w'))
        print(f'{name}: DONE. Results saved to {d}')
    finally:
        if mon:
            mon.halt = True
        stop(p)

## 5. Run the variants
One cell per variant, so a failure in one does not stop the others. Each takes about 10 minutes and prints a line per
concurrency level. If a server fails to start, the error shows the last lines of its log: paste them to Claude.

In [ ]:
run_variant('fp16')

In [ ]:
run_variant('awq-default')

In [ ]:
run_variant('awq-mlp-only')

In [ ]:
run_variant('gguf-q4km')

In [ ]:
run_variant('gguf-q8')

## 6. Results and cost
**Cost per 1M requests** = (GPU price per hour) / (requests per second x 3600) x 1,000,000, using the throughput at the best concurrency level.
That assumes the GPU is kept busy, so it is a best-case figure for each variant.
Set `HOURLY` to your real GPU price. Colab does not publish one hourly price, so the table also shows the cost at a few example rates.
To compare against a hosted API, fill in `HOSTED_IN` and `HOSTED_OUT` (dollars per 1M tokens) from your provider's current price page; they are left empty on purpose, so nothing here is guessed.

In [ ]:
import pandas as pd
HOURLY = 1.20                       # <- edit: your GPU $/hour
EXAMPLE_RATES = [1.0, 2.0, 4.0]     # $/hour, for a feel of how cost scales
HOSTED_IN, HOSTED_OUT = None, None  # <- optional: hosted API $ per 1M input / output tokens

sweep = pd.read_csv(f'{RES}/sweep.csv')
rows = []
for name in [n for n, _, _ in SPECS]:
    d = f'{RES}/{name}'
    if not (os.path.exists(f'{d}/finetuned_test.json') and os.path.exists(f'{d}/serve_info.json')):
        continue
    q, info = json.load(open(f'{d}/finetuned_test.json')), json.load(open(f'{d}/serve_info.json'))
    s = sweep[sweep.variant == name]
    pk = s.loc[s.req_per_s.idxmax()]
    c1, c32 = s[s.concurrency == 1].iloc[0], s[s.concurrency == 32].iloc[0]
    row = {'variant': name, 'engine': info['engine'], 'weights_GiB': info['weights_gib'],
           'intent_acc': round(q['intent_accuracy'], 4), 'macro_F1': round(q['intent_macro_f1'], 4),
           'invented_%': round(100 * q['invented_label_rate'], 2),
           'p50_c1_ms': round(1000 * c1.p50_s), 'p50_c32_ms': round(1000 * c32.p50_s), 'p95_c32_ms': round(1000 * c32.p95_s),
           'req/s_c32': c32.req_per_s, 'peak_req/s': pk.req_per_s, 'peak_at_conc': int(pk.concurrency), 'peak_tok/s': pk.out_tok_per_s,
           'errors': int(s.errors.sum())}
    row[f'$/1M req @${HOURLY}/h'] = round(HOURLY / (pk.req_per_s * 3600) * 1e6, 2)
    for r in EXAMPLE_RATES:
        row[f'$/1M req @${r}/h'] = round(r / (pk.req_per_s * 3600) * 1e6, 2)
    row[f'$/1M out tok @${HOURLY}/h'] = round(HOURLY / (pk.out_tok_per_s * 3600) * 1e6, 2)
    row['mean_prompt_tok'], row['mean_out_tok'] = pk.mean_prompt_tokens, pk.mean_out_tokens
    if HOSTED_IN is not None and HOSTED_OUT is not None:
        row['hosted $/1M req'] = round(pk.mean_prompt_tokens * HOSTED_IN + pk.mean_out_tokens * HOSTED_OUT, 2)
    rows.append(row)

summary = pd.DataFrame(rows)
summary.to_csv(f'{RES}/serving_summary.csv', index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
display(summary)

In [ ]:
# Plots: throughput and tail latency vs. number of simultaneous requests.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name in summary.variant:
    s = sweep[sweep.variant == name].sort_values('concurrency')
    ax[0].plot(s.concurrency, s.req_per_s, marker='o', label=name)
    ax[1].plot(s.concurrency, s.p95_s * 1000, marker='o', label=name)
for a, t, y in [(ax[0], 'Throughput', 'requests / second'), (ax[1], 'Tail latency (p95)', 'milliseconds')]:
    a.set_xscale('log', base=2); a.set_xlabel('simultaneous requests'); a.set_ylabel(y); a.set_title(t); a.grid(alpha=0.3)
ax[1].set_yscale('log'); ax[0].legend()
plt.tight_layout(); plt.savefig(f'{RES}/serving_curves.png', dpi=130); plt.show()
print('saved to', RES)

## 7. Quality check through the serving stack
The accuracy of every variant when scored through its server (all 3,080 examples). It should match the Phase 3 numbers
(fp16 0.9373, GGUF Q4_K_M 0.9357, AWQ default 0.9312). A big gap would mean the server is processing prompts differently.

In [ ]:
cols = ['variant', 'engine', 'intent_acc', 'macro_F1', 'invented_%', 'errors']
display(summary[cols])
PHASE3 = {'fp16': 0.9373, 'gguf-q8': 0.9383, 'gguf-q4km': 0.9357, 'awq-default': 0.9312, 'awq-mlp-only': 0.9331}
for _, r in summary.iterrows():
    ref = PHASE3.get(r.variant)
    if ref is not None:
        print(f'{r.variant:<14} serving {r.intent_acc:.4f} vs Phase 3 {ref:.4f}  (diff {r.intent_acc - ref:+.4f})')

## 8. Save a copy
Everything is already on Drive in `Banking_FInetuning_phase4/` (`serving_summary.csv`, `sweep.csv`, `serving_curves.png`, and one folder per variant with its server log, scores and answers). This cell also downloads a zip to your computer.

In [ ]:
shutil.make_archive('/content/phase4_results', 'zip', RES)
print(sh('ls -lh /content/phase4_results.zip'))
files.download('/content/phase4_results.zip')